In [ ]:
!pip install --upgrade torch transformers==4.45.0 huggingface_hub accelerate numpy<2.0

In [ ]:
import os
os.environ["WANDB_DISABLED"] = "true"
import logging
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
import re
from datasets import load_dataset, Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
    DataCollatorWithPadding
)
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report
import warnings

warnings.filterwarnings("ignore")

# Setup logging
logging.basicConfig(
    format="%(asctime)s - %(levelname)s - %(name)s - %(message)s",
    datefmt="%m/%d/%Y %H:%M:%S",
    level=logging.INFO,
)
logger = logging.getLogger(__name__)
print(f"Torch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")


In [ ]:
class FocalLoss(nn.Module):
    def __init__(self, alpha=1, gamma=2, reduction='mean'):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, reduction='none')
        pt = torch.exp(-ce_loss)
        focal_loss = self.alpha * (1 - pt) ** self.gamma * ce_loss
        if self.reduction == 'mean':
            return focal_loss.mean()
        elif self.reduction == 'sum':
             return focal_loss.sum()
        return focal_loss

In [ ]:
import os
os.environ["WANDB_DISABLED"] = "true"
# Set TensorBoard logging directory (replaces deprecated logging_dir)
os.environ["TENSORBOARD_DIR"] = "./logs"

import logging
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
import re
from datasets import load_dataset, Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
    DataCollatorWithPadding
)
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report
from huggingface_hub import HfApi
import warnings

warnings.filterwarnings("ignore")

# Setup logging
logging.basicConfig(
    format="%(asctime)s - %(levelname)s - %(name)s - %(message)s",
    datefmt="%m/%d/%Y %H:%M:%S",
    level=logging.INFO,
)
logger = logging.getLogger(__name__)


class FocalLoss(nn.Module):
    """Focal loss for handling class imbalance."""
    def __init__(self, alpha=1, gamma=2, reduction='mean'):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, reduction='none')
        pt = torch.exp(-ce_loss)
        focal_loss = self.alpha * (1 - pt) ** self.gamma * ce_loss
        if self.reduction == 'mean':
            return focal_loss.mean()
        elif self.reduction == 'sum':
            return focal_loss.sum()
        return focal_loss


class CodeDetectionTrainer:
    """High-level trainer for machine-generated code detection using Trainer API."""
    
    def __init__(self, max_length=512, model_name="microsoft/graphcodebert-base", seed=42, use_focal_loss=True):
        self.max_length = max_length
        self.model_name = model_name
        self.seed = seed
        self.use_focal_loss = use_focal_loss
        self.tokenizer = None
        self.model = None
        self.trainer = None
        self.num_labels = None
        
    def load_and_prepare_data(self):
        """Load datasets from Hugging Face Hub."""
        logger.info("Loading datasets from Hugging Face Hub...")
        try:
            train_dataset = load_dataset("DaniilOr/SemEval-2026-Task13", "A", split="train")
            val_dataset = load_dataset("DaniilOr/SemEval-2026-Task13", "A", split="validation")

            if 'code' not in train_dataset.column_names or 'label' not in train_dataset.column_names:
                raise ValueError("Dataset must contain 'code' and 'label' columns")

            self.num_labels = len(set(train_dataset['label']))
            logger.info(f"Train samples: {len(train_dataset)}, Validation samples: {len(val_dataset)}")
            logger.info(f"Number of labels: {self.num_labels}")
            
            return train_dataset, val_dataset
        except Exception as e:
            logger.error(f"Error loading dataset: {e}")
            raise

    def preprocess_code(self, code_str):
        """Preprocess code string by normalizing whitespace."""
        code_str = re.sub(r'[\r\n]+', '\n', code_str)
        code_str = re.sub(r'[ \t]+', ' ', code_str)
        return code_str

    def initialize_model_and_tokenizer(self):
        """Initialize model and tokenizer."""
        logger.info(f"Initializing {self.model_name}...")
        self.tokenizer = AutoTokenizer.from_pretrained(self.model_name)
        self.model = AutoModelForSequenceClassification.from_pretrained(
            self.model_name,
            num_labels=self.num_labels,
            problem_type="single_label_classification"
        )
        logger.info(f"Model initialized with {self.num_labels} labels")

    def tokenize_function(self, examples):
        """Tokenize code snippets."""
        cleaned_codes = [self.preprocess_code(c) for c in examples['code']]
        return self.tokenizer(
            cleaned_codes,
            truncation=True,
            max_length=self.max_length,
            padding=True,
        )

    def prepare_datasets(self, train_dataset, val_dataset):
        """Prepare datasets for training."""
        logger.info("Preparing datasets...")
        
        # Columns to remove after tokenization
        columns_to_remove = ['code', 'generator', 'language']
        columns_to_remove = [col for col in columns_to_remove if col in train_dataset.column_names]
        
        # Tokenize
        train_dataset = train_dataset.map(
            self.tokenize_function, 
            batched=True, 
            remove_columns=columns_to_remove
        )
        val_dataset = val_dataset.map(
            self.tokenize_function, 
            batched=True, 
            remove_columns=columns_to_remove
        )
        
        # Rename label column
        train_dataset = train_dataset.rename_column('label', 'labels')
        val_dataset = val_dataset.rename_column('label', 'labels')
        
        return train_dataset, val_dataset

    def compute_metrics(self, eval_pred):
        """Compute macro F1 score (primary metric) and other metrics."""
        predictions, labels = eval_pred
        predictions = np.argmax(predictions, axis=1)
        
        # Compute macro F1 (primary metric)
        macro_f1 = precision_recall_fscore_support(labels, predictions, average='macro')[2]
        
        # Compute other metrics
        accuracy = accuracy_score(labels, predictions)
        precision, recall, f1_weighted, _ = precision_recall_fscore_support(
            labels, predictions, average='weighted'
        )
        
        return {
            'macro_f1': macro_f1,
            'accuracy': accuracy,
            'f1_weighted': f1_weighted,
            'precision': precision,
            'recall': recall
        }

    def train(self, output_dir="./results", num_epochs=3, batch_size=16, learning_rate=2e-5):
        """Train the model using the Trainer API."""
        logger.info("Loading data...")
        train_dataset, val_dataset = self.load_and_prepare_data()
        
        logger.info("Initializing model...")
        self.initialize_model_and_tokenizer()
        
        logger.info("Preparing datasets...")
        train_dataset, val_dataset = self.prepare_datasets(train_dataset, val_dataset)
        
        # Calculate warmup steps (10% of total steps)
        num_update_steps_per_epoch = len(train_dataset) // batch_size
        total_training_steps = num_epochs * num_update_steps_per_epoch
        warmup_steps = int(total_training_steps * 0.1)
        
        # Configure training arguments
        training_args = TrainingArguments(
            output_dir=output_dir,
            num_train_epochs=num_epochs,
            per_device_train_batch_size=batch_size,
            per_device_eval_batch_size=batch_size,
            weight_decay=0.01,
            logging_steps=10,
            evaluation_strategy="steps",
            eval_steps=500,
            save_strategy="steps",
            save_steps=500,
            load_best_model_at_end=True,
            metric_for_best_model="macro_f1",  # Optimize for macro F1
            greater_is_better=True,
            remove_unused_columns=False,
            learning_rate=learning_rate,
            lr_scheduler_type="linear",
            warmup_steps=warmup_steps,  # Use warmup_steps instead of deprecated warmup_ratio
            save_total_limit=2,
            report_to=[],  # Disable W&B
            seed=self.seed,
        )
        
        # Initialize trainer with custom loss if needed
        data_collator = DataCollatorWithPadding(tokenizer=self.tokenizer)
        
        # Note: tokenizer parameter is NOT passed to Trainer in newer versions
        self.trainer = Trainer(
            model=self.model,
            args=training_args,
            train_dataset=train_dataset,
            eval_dataset=val_dataset,
            data_collator=data_collator,
            compute_metrics=self.compute_metrics,
            callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
        )
        
        # Optionally replace loss function with focal loss
        if self.use_focal_loss:
            logger.info("Using Focal Loss for training")
            focal_loss_fn = FocalLoss(alpha=1, gamma=2)
            
            # Override compute_loss to use focal loss
            def compute_loss_with_focal(model, inputs, return_outputs=False):
                labels = inputs.pop("labels")
                outputs = model(**inputs)
                logits = outputs.logits
                loss = focal_loss_fn(logits, labels)
                return (loss, outputs) if return_outputs else loss
            
            self.trainer.compute_loss = compute_loss_with_focal
        
        logger.info("***** Starting training *****")
        logger.info(f"  Max epochs: {num_epochs}")
        logger.info(f"  Batch size per device: {batch_size}")
        logger.info(f"  Learning rate: {learning_rate}")
        logger.info(f"  Warmup steps: {warmup_steps}")
        logger.info(f"  Total training steps: {total_training_steps}")
        logger.info(f"  Using focal loss: {self.use_focal_loss}")
        
        self.trainer.train()
        
        logger.info(f"Training completed. Model saved to {output_dir}")
        return self.trainer

    def evaluate(self, eval_dataset=None):
        """Evaluate the model and print detailed report."""
        if self.trainer is None:
            logger.error("No trainer found. Train the model first.")
            return None
        
        logger.info("Evaluating model...")
        predictions = self.trainer.predict(eval_dataset)
        y_pred = np.argmax(predictions.predictions, axis=1)
        y_true = predictions.label_ids
        
        logger.info("\n***** Classification Report *****")
        print(classification_report(y_true, y_pred, digits=4))
        
        return predictions

    def run_full_pipeline(self, output_dir="./results", num_epochs=3, batch_size=16, learning_rate=2e-5):
        """Run complete training and evaluation pipeline."""
        try:
            self.train(output_dir, num_epochs, batch_size, learning_rate)
            logger.info(f"Best model saved to: {os.path.join(output_dir, 'best_model')}")
            return self.trainer
        except Exception as e:
            logger.error(f"Pipeline error: {e}")
            raise


In [ ]:
OUTPUT_DIR = "taskA-graphcodebert-focal"

# Initialize trainer
trainer_obj = CodeDetectionTrainer(
    max_length=512,
    model_name="microsoft/graphcodebert-base",
    seed=42,
    use_focal_loss=True  # Enable focal loss for better handling of class imbalance
)

# Run training pipeline
trainer_obj.run_full_pipeline(
    output_dir=OUTPUT_DIR,
    num_epochs=10,
    batch_size=16,
    learning_rate=2e-5
)


In [ ]:
# Setup HuggingFace authentication
from huggingface_hub import login

# Option 1: Login with your HuggingFace token (run this once)
# Uncomment and run if you haven't set up authentication yet
# login()  # This will prompt you to enter your HuggingFace API token

# Option 2: Set HF_TOKEN environment variable (recommended for notebooks)
# Uncomment and replace with your actual token if needed:
# os.environ["HF_TOKEN"] = "your_huggingface_token_here"

# Check if authenticated
try:
    from huggingface_hub import get_whoami
    user_info = get_whoami()
    logger.info(f"Authenticated as: {user_info['name']}")
except:
    logger.warning("Not authenticated with HuggingFace. Checkpoints will not be uploaded.")
    logger.info("To authenticate, run: from huggingface_hub import login; login()")
    logger.info("Or set HF_TOKEN environment variable with your API token from https://huggingface.co/settings/tokens")


In [ ]:
from itertools import chain
from tqdm.auto import tqdm
import re

def preprocess_code(code_str):
    """Preprocess code string."""
    code_str = re.sub(r'[\r\n]+', '\n', code_str)
    code_str = re.sub(r'[ \t]+', ' ', code_str)
    return code_str

@torch.no_grad()
def predict_on_test(checkpoint_dir, output_path, max_length=512, batch_size=16):
    """
    Load model from checkpoint and perform predictions on test set.
    
    Args:
        checkpoint_dir: Path to the checkpoint directory containing model files
        output_path: Path to save predictions CSV
        max_length: Maximum sequence length for tokenization
        batch_size: Batch size for inference
    """
    device = "cuda" if torch.cuda.is_available() else "cpu"
    
    logger.info(f"Loading model and tokenizer from: {checkpoint_dir}")
    
    # Load model and tokenizer from checkpoint
    try:
        model = AutoModelForSequenceClassification.from_pretrained(checkpoint_dir)
        tokenizer = AutoTokenizer.from_pretrained(checkpoint_dir)
    except Exception as e:
        logger.error(f"Error loading checkpoint from {checkpoint_dir}: {e}")
        raise
    
    model = model.to(device)
    model.eval()
    logger.info(f"Model loaded on device: {device}")

    # Load test dataset from Hugging Face Hub
    logger.info("Loading test dataset...")
    ds = load_dataset("DaniilOr/SemEval-2026-Task13", "A", split="test", streaming=True)
    it = iter(ds)
    first = next(it)
    stream = chain([first], it)

    def batcher(iterator, bs):
        """Helper function to create batches from iterator."""
        buf = []
        for ex in iterator:
            buf.append(ex)
            if len(buf) == bs:
                yield buf
                buf = []
        if buf:
            yield buf

    # Perform inference on test set
    with open(output_path, "w") as f:
        f.write("id,label\n")
        logger.info("Predicting on test set...")

        for batch in tqdm(batcher(stream, batch_size), desc="Predicting"):
            # Preprocess codes
            codes = [preprocess_code(row["code"]) for row in batch]
            ids = [row["id"] for row in batch]

            # Tokenize
            enc = tokenizer(
                codes,
                truncation=True,
                padding=True,
                max_length=max_length,
                return_tensors="pt",
            )
            input_ids = enc["input_ids"].to(device)
            attention_mask = enc["attention_mask"].to(device)

            # Get predictions
            logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
            pred_labels = logits.argmax(dim=-1).cpu().tolist()

            # Write to CSV
            for ex_id, pred in zip(ids, pred_labels):
                f.write(f"{ex_id},{pred}\n")

    logger.info(f"Predictions saved to {output_path}")


# ============== Prediction Configuration ==============
# Choose which checkpoint to use for prediction:
# Option 1: Use the best model checkpoint (highest macro F1)
CHECKPOINT_PATH = "taskA-graphcodebert-focal/best_model"

# Option 2: Use the latest checkpoint
# CHECKPOINT_PATH = "taskA-graphcodebert-focal/checkpoint-<step>"

OUT_CSV = "submission.csv"

# Run prediction using checkpoint
predict_on_test(
    checkpoint_dir=CHECKPOINT_PATH,
    output_path=OUT_CSV,
    max_length=512,
    batch_size=32,
)

logger.info(f"Predictions written to: {OUT_CSV}")
